# Дистилляция: Qwen2.5-1.5B учится на ответах Qwen2.5-7B

Два адаптера на одних и тех же входах корпуса:

- `distill` — 312 меток учителя, отфильтрованных только по **форме** (честная
  дистилляция, ученик копирует учителя «как есть»);
- `golden` — 211 меток, совпавших со **смыслом** (строгая чистка).

Данные уже собраны локально (`make corpus / label / filter`). Ноутбук попросит
`distillation.zip`, обучит оба варианта и упакует адаптеры + `run_params.json`
с путями репозитория: локально `unzip -o distillation_results.zip -d <repo>`
восстановит `finetuning/`.

Включите T4 и выполняйте ячейки по порядку.

In [ ]:
!pip -q install -U "transformers>=4.56" "peft>=0.13" accelerate matplotlib
!pip -q uninstall -y torchao
# После этой ячейки: Среда выполнения → Перезапустить сеанс, затем выполнять со следующей.

In [ ]:
import glob
import os
import subprocess

from google.colab import files

os.chdir('/content')
project = '/content/finetuning/distillation'

if not os.path.exists(f'{project}/data/train_distill.jsonl'):
    print('Выберите файл distillation.zip')
    files.upload()
    zips = sorted(glob.glob('/content/distillation*.zip'))
    if not zips:
        raise RuntimeError('zip не найден в /content')
    subprocess.run(['unzip', '-qo', zips[-1], '-d', '/content'], check=True)

os.chdir(project)
print('проект:', os.getcwd())
for name in ('train_distill.jsonl', 'valid_distill.jsonl', 'train_golden.jsonl', 'valid_golden.jsonl'):
    print(name, os.path.exists(f'data/{name}'))

In [ ]:
import json

for variant in ('distill', 'golden'):
    train = [json.loads(line) for line in open(f'data/train_{variant}.jsonl', encoding='utf-8') if line.strip()]
    valid = [json.loads(line) for line in open(f'data/valid_{variant}.jsonl', encoding='utf-8') if line.strip()]
    print(f'{variant}: train={len(train)} valid={len(valid)}')
    example = train[0]['messages']
    print('  вход :', example[1]['content'])
    print('  метка:', example[2]['content'])

## Обучение (2 эпохи, LoRA r=16/α=32)

Каждое обучение ~10–15 минут на T4. `run_params.json` и `loss.png` пишутся в
`/content/run_distill` и `/content/run_golden`.

In [ ]:
!python training/train_lora.py --variant distill --out /content/adapter_distill --run-dir /content/run_distill

In [ ]:
!python training/train_lora.py --variant golden --out /content/adapter_golden --run-dir /content/run_golden

In [ ]:
import json

from IPython.display import Image, display

for variant in ('distill', 'golden'):
    run = json.load(open(f'/content/run_{variant}/run_params.json', encoding='utf-8'))
    evals = run.get('eval_loss_history') or []
    evals_txt = f'{evals[0][1]:.3f} -> {evals[-1][1]:.3f}' if evals else '—'
    print(f'{variant}: train loss {run["loss_first"]:.3f} -> {run["loss_last"]:.3f} | ')
    print(f'         valid loss {evals_txt}')
    display(Image(f'/content/run_{variant}/loss.png'))

## Упаковать результат

Архив повторяет пути репозитория. Локально: `unzip -o distillation_results.zip -d <repo>`
даст `finetuning/distillation/results/` (обе `run_*_params.json` и `loss.png`) и
`finetuning/artifacts/distillation/adapter_distill|adapter_golden`.

Дальше локально: слить адаптеры → GGUF → `ollama create qwen25-distill-v1` и
`qwen25-golden-v1` → `make eval-all && make compare` (см. `README.md`).

In [ ]:
import os
import shutil

artifacts = '/content/finetuning/artifacts/distillation'
results = '/content/finetuning/distillation/results'
os.makedirs(artifacts, exist_ok=True)
os.makedirs(results, exist_ok=True)

for variant in ('distill', 'golden'):
    dst = f'{artifacts}/adapter_{variant}'
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(f'/content/adapter_{variant}', dst)
    shutil.copy2(f'/content/run_{variant}/run_params.json', f'{results}/run_{variant}_params.json')
    shutil.copy2(f'/content/run_{variant}/loss.png', f'{results}/run_{variant}_loss.png')
    print('сохранено:', variant)

!cd /content && rm -f distillation_results.zip
!cd /content && zip -qr distillation_results.zip finetuning/distillation/results finetuning/artifacts/distillation

from google.colab import files

files.download('/content/distillation_results.zip')